In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# -------------------------------
# Hybrid Activation
# -------------------------------
class HybridActivation(nn.Module):
    """
    A simple hybrid activation that averages ReLU and ELU outputs.
    You can modify the weighting or functions as needed.
    """
    def __init__(self, alpha=0.5):
        super(HybridActivation, self).__init__()
        self.alpha = alpha

    def forward(self, x):
        return self.alpha * F.relu(x) + (1 - self.alpha) * F.elu(x)

# -------------------------------
# Inception-style Dense Layer with Hybrid Activations
# -------------------------------
class InceptionDenseLayer(nn.Module):
    """
    A single dense layer that implements an Inception-style block with a bottleneck.

    The module performs:
      1. BatchNorm and HybridActivation on the input.
      2. A 1×1 convolution (bottleneck) reducing the number of channels.
      3. Two parallel convolution branches:
           - Branch A: 3×3 convolution (with padding=1)
           - Branch B: 5×5 convolution (with padding=2)
         Each branch produces a fraction of the growth rate.
      4. The outputs of the two branches are concatenated.
      5. (Optional) Dropout is applied.
      6. Finally, the result is concatenated with the original input (dense connectivity).
    """
    def __init__(self, num_input_features, growth_rate, bn_size, drop_rate):
        super(InceptionDenseLayer, self).__init__()
        inter_channels = bn_size * growth_rate  # bottleneck channels

        self.norm = nn.BatchNorm2d(num_input_features)
        self.hybrid_act = HybridActivation(alpha=0.5)
        self.conv1 = nn.Conv2d(num_input_features, inter_channels, kernel_size=1, stride=1, bias=False)

        # Split growth rate across branches
        branch_a_channels = growth_rate // 2
        branch_b_channels = growth_rate - branch_a_channels

        # Branch A: 3x3 conv
        self.branch_a = nn.Conv2d(inter_channels, branch_a_channels,
                                  kernel_size=3, stride=1, padding=1, bias=False)
        # Branch B: 5x5 conv (using padding=2)
        self.branch_b = nn.Conv2d(inter_channels, branch_b_channels,
                                  kernel_size=5, stride=1, padding=2, bias=False)
        self.drop_rate = drop_rate

    def forward(self, x):
        # Apply normalization and hybrid activation
        out = self.norm(x)
        out = self.hybrid_act(out)
        # Bottleneck convolution
        out = self.conv1(out)
        # Two parallel branches
        out_a = self.branch_a(out)
        out_b = self.branch_b(out)
        # Concatenate branch outputs along channel dimension
        out = torch.cat([out_a, out_b], dim=1)
        if self.drop_rate > 0:
            out = F.dropout(out, p=self.drop_rate, training=self.training)
        # Dense connectivity: concatenate input and new features
        out = torch.cat([x, out], dim=1)
        return out

# -------------------------------
# Inception Dense Block
# -------------------------------
class InceptionDenseBlock(nn.Module):
    """
    A block that stacks multiple InceptionDenseLayer modules.

    Each new layer receives as input all preceding feature maps (dense connectivity).
    """
    def __init__(self, num_layers, num_input_features, growth_rate, bn_size, drop_rate):
        super(InceptionDenseBlock, self).__init__()
        layers = []
        for i in range(num_layers):
            layer = InceptionDenseLayer(
                num_input_features + i * growth_rate,
                growth_rate=growth_rate,
                bn_size=bn_size,
                drop_rate=drop_rate
            )
            layers.append(layer)
        self.layers = nn.Sequential(*layers)

    def forward(self, x):
        return self.layers(x)

# -------------------------------
# Transition Layer (Same as in DenseNet)
# -------------------------------
class _Transition(nn.Module):
    """
    A transition layer that reduces the number of channels and downsamples
    the spatial dimensions (using a 1×1 convolution followed by 2×2 average pooling).
    """
    def __init__(self, num_input_features, num_output_features):
        super(_Transition, self).__init__()
        self.norm = nn.BatchNorm2d(num_input_features)
        self.relu = nn.ReLU(inplace=True)
        self.conv = nn.Conv2d(num_input_features, num_output_features,
                              kernel_size=1, stride=1, bias=False)
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        out = self.conv(self.relu(self.norm(x)))
        out = self.pool(out)
        return out

# -------------------------------
# Inception DenseNet Model
# -------------------------------
class InceptionDenseNet(nn.Module):
    """
    An Inception DenseNet model that follows the ideas from
    “Inception DenseNet With Hybrid Activations For Image Classification.”

    The network starts with an initial convolution (and pooling) layer, followed by a series of
    inception dense blocks with transition layers in between (except after the last block).
    After a final batch normalization and ReLU, global average pooling is applied before
    the final classifier.
    """
    def __init__(self, growth_rate=32, block_config=(6, 12, 24, 16),
                 num_init_features=64, bn_size=4, drop_rate=0, num_classes=1000):
        super(InceptionDenseNet, self).__init__()
        # Initial convolution and pooling (stem)
        self.features = nn.Sequential(
            nn.Conv2d(3, num_init_features, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(num_init_features),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1)
        )
        # Build Inception Dense Blocks with Transition Layers
        num_features = num_init_features
        self.blocks = nn.ModuleList()
        self.transitions = nn.ModuleList()
        for i, num_layers in enumerate(block_config):
            block = InceptionDenseBlock(num_layers, num_features, growth_rate, bn_size, drop_rate)
            self.blocks.append(block)
            num_features = num_features + num_layers * growth_rate
            if i != len(block_config) - 1:
                trans = _Transition(num_features, num_features // 2)
                self.transitions.append(trans)
                num_features = num_features // 2

        # Final batch norm
        self.norm_final = nn.BatchNorm2d(num_features)
        # Classifier
        self.classifier = nn.Linear(num_features, num_classes)

        # Weight initialization (optional)
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        # Stem
        out = self.features(x)
        # Dense blocks with transitions
        for i, block in enumerate(self.blocks):
            out = block(out)
            if i < len(self.transitions):
                out = self.transitions[i](out)
        # Final batch norm and ReLU
        out = self.norm_final(out)
        out = F.relu(out, inplace=True)
        # Global average pooling and classifier
        out = F.adaptive_avg_pool2d(out, (1, 1)).view(out.size(0), -1)
        out = self.classifier(out)
        return out

# -------------------------------
# Testing the Inception DenseNet Model
# -------------------------------
if __name__ == '__main__':
    # Create an instance of InceptionDenseNet (example configuration)
    model = InceptionDenseNet(
        growth_rate=32,
        block_config=(6, 12, 24, 16),  # you can change this tuple to modify depth
        num_init_features=64,
        bn_size=4,
        drop_rate=0.2,
        num_classes=10
    )
    # Dummy input: batch of 2 images with 3 channels and 224x224 resolution
    x = torch.randn(2, 3, 224, 224)
    y = model(x)
    print("Output shape:", y.shape)  # Expected output shape: (2, 10)


Output shape: torch.Size([2, 10])
